# Laboratorio 10
### Aprendizaje Semi-supervisado.
- Fabian Prado #23427
- Sofia Lopez #231929
- Jonathan Zacarias #231104
---

## Analisis Exploratorio

Primero se realiza la importación de librerias, se carga el dataset se inspeccionan los datos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)


Librerías cargadas correctamente.


In [ ]:
df = pd.read_csv('data/bank-full.csv', sep=';')

print(f'Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(df.dtypes.value_counts())

Dimensiones: 45,211 filas x 17 columnas


### Descripción de los variables

| Variable | Tipo | Descripción |
|---|---|---|
| `age` | Numérica | Edad del cliente |
| `job` | Categórica | Tipo de trabajo |
| `marital` | Categórica | Estado civil |
| `education` | Categórica | Nivel educativo |
| `default` | Binaria | ¿Tiene crédito en mora? |
| `balance` | Numérica | Saldo promedio anual (euros) |
| `housing` | Binaria | ¿Tiene préstamo hipotecario? |
| `loan` | Binaria | ¿Tiene préstamo personal? |
| `contact` | Categórica | Tipo de contacto |
| `day` | Numérica | Día del mes del último contacto |
| `month` | Categórica | Mes del último contacto |
| `duration` | Numérica | Duración del último contacto (segundos) |
| `campaign` | Numérica | Número de contactos en esta campaña |
| `pdays` | Numérica | Días desde contacto de campaña anterior (-1 = nunca) |
| `previous` | Numérica | Contactos antes de esta campaña |
| `poutcome` | Categórica | Resultado de la campaña anterior |
| `y` | Binaria| ¿Suscribió un depósito a plazo? (target) |

In [ ]:
print('== Tipos de datos ==')
print(df.dtypes)
print(f'\nVariables numéricas  : {df.select_dtypes(include="number").columns.tolist()}')
print(f'Variables categóricas: {df.select_dtypes(include="str").columns.tolist()}')


== Tipos de datos ==
age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
y              str
dtype: object

Variables numéricas  : ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Variables categóricas: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'y']
== Estadísticas descriptivas – Variables numéricas ==
== Estadísticas descriptivas – Variables categóricas ==


,job,marital,education,default,housing,loan,contact,month,poutcome,y
count,45211,45211,45211,45211,45211,45211,45211,45211,45211,45211
unique,12,3,4,2,2,2,3,12,4,2
top,blue-collar,married,secondary,no,yes,no,cellular,may,unknown,no
freq,9732,27214,23202,44396,25130,37967,29285,13766,36959,39922


In [13]:
print('== Estadísticas descriptivas – Variables numéricas ==')
df.describe().round(2)

== Estadísticas descriptivas – Variables numéricas ==


,age,balance,day,duration,campaign,pdays,previous
count,45211.00,45211.00,45211.00,45211.00,45211.00,45211.00,45211.00
mean,40.94,1362.27,15.81,258.16,2.76,40.20,0.58
std,10.62,3044.77,8.32,257.53,3.10,100.13,2.30
min,18.00,-8019.00,1.00,0.00,1.00,-1.00,0.00
25%,33.00,72.00,8.00,103.00,1.00,-1.00,0.00
50%,39.00,448.00,16.00,180.00,2.00,-1.00,0.00
75%,48.00,1428.00,21.00,319.00,3.00,-1.00,0.00
max,95.00,102127.00,31.00,4918.00,63.00,871.00,275.00


In [14]:
print('== Estadísticas descriptivas – Variables categóricas ==')
df.describe(include='str')

== Estadísticas descriptivas – Variables categóricas ==


,job,marital,education,default,housing,loan,contact,month,poutcome,y
count,45211,45211,45211,45211,45211,45211,45211,45211,45211,45211
unique,12,3,4,2,2,2,3,12,4,2
top,blue-collar,married,secondary,no,yes,no,cellular,may,unknown,no
freq,9732,27214,23202,44396,25130,37967,29285,13766,36959,39922


### Limpieza de datos

In [15]:
nulos = df.isnull().sum()
print('Valores nulos por columna:')
print(nulos)
print(f'\nTotal de nulos: {nulos.sum()}')

Valores nulos por columna:
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

Total de nulos: 0


El dataset **no presenta valores nulos explícitos**. Sin embargo, varias columnas usan `"unknown"` como marcador implícito de dato faltante.

In [ ]:
cat_cols = df.select_dtypes(include='str').columns.tolist()

unknowns = {col: (df[col] == 'unknown').sum() for col in cat_cols}
unknowns_df = pd.DataFrame.from_dict(unknowns, orient='index', columns=['count_unknown'])
unknowns_df['pct (%)'] = (unknowns_df['count_unknown'] / len(df) * 100).round(2)
unknowns_df = unknowns_df[unknowns_df['count_unknown'] > 0]
print('Columnas con valores "unknown":')
print(unknowns_df)

Columnas con valores "unknown":
           count_unknown  pct (%)
job                  288     0.64
education           1857     4.11
contact            13020    28.80
poutcome           36959    81.75


C:\Users\sofia\AppData\Local\Temp\ipykernel_24232\840023419.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()
